In [6]:
import os
import s3fs
import pandas as pd

In [7]:
S3_ENDPOINT_URL = "https://" + os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL

'https://minio.lab.sspcloud.fr'

In [8]:
fs = s3fs.S3FileSystem(client_kwargs= {'endpoint_url' : S3_ENDPOINT_URL})
# fs.ls('donnees-insee') pour vérifier

In [11]:
BUCKET = '1020cepe02'
FILE_KEY_S3 = 'data/readmission_avc.parquet'
FILE_PATH_S3 = BUCKET + '/' + FILE_KEY_S3
FILE_KEY_S3
# dossier data sur onyxia / explorateur de fichiers / mes données / data

'data/readmission_avc.parquet'

In [47]:
with fs.open(FILE_PATH_S3, mode = 'rb') as file_in :
    dataini = pd.read_parquet(file_in)

In [48]:
dataini
# lignes : séjour hospitalier
# prédire si un patient revient sous 30 jours à l'hopital
# id_D : identifiant du prochain séjour. Si None , sujet pas revenu la fois suivante
# catégoriel : modeEntrée et modeSortie, dp (diagnostic principal), sexe, ghm2 (complexité / type de séjour )

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,
3,8,8,11,01M301,I639,1.0,83.0,4,2,2.0,8oi,None
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld
...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,


## 1. Data preprocessing

In [49]:
# Taille
dataini.shape

(1700, 12)

In [50]:
# Données manquantes
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           20
age            20
nbActe          0
nbRum           0
nbda          134
id              0
id_D          200
dtype: int64

### 1.1 Encoding label (Y)

In [51]:
dataini = dataini.dropna(subset=['id_D'], axis = 0)

In [52]:
dataini['rea'] = (dataini['id_D'] != '').astype('int8')

/tmp/ipykernel_5794/253238662.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataini['rea'] = (dataini['id_D'] != '').astype('int8')


In [53]:
dataini['rea'].value_counts()

rea
0    1281
1     219
Name: count, dtype: int64

In [54]:
# a priori dataset peu déséquilibré

### 1.2 Categorical variables

In [55]:
# on regarde les X
dataini.dtypes

modeEntree      int32
modeSortie      int32
duree           int32
ghm2           object
dp             object
sexe          float64
age           float64
nbActe          int32
nbRum           int32
nbda          float64
id             object
id_D           object
rea              int8
dtype: object

In [56]:
str_cols = ['modeEntree', 'modeSortie', 'sexe']


In [60]:
dataini[str_cols] = dataini[str_cols].astype('object')
# attention objcet ou str ne donnent pas les mêmes encodages
# avec str : on ne voit plus les données manquantes de sexe : nan est considéré comme une modalité caractère

/tmp/ipykernel_5794/1904198915.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataini[str_cols] = dataini[str_cols].astype('object')


In [61]:
dataini.dtypes

modeEntree     object
modeSortie     object
duree           int32
ghm2           object
dp             object
sexe           object
age           float64
nbActe          int32
nbRum           int32
nbda          float64
id             object
id_D           object
rea              int8
dtype: object

In [62]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           19
age            19
nbActe          0
nbRum           0
nbda          121
id              0
id_D            0
rea             0
dtype: int64

### 1.3 Faux manquant

In [63]:
dataini.nbda.value_counts()

nbda
3.0     193
4.0     178
2.0     154
1.0     150
5.0     135
6.0     129
7.0     115
8.0      85
9.0      68
10.0     37
11.0     32
13.0     27
12.0     20
14.0     14
15.0     13
16.0      8
17.0      4
18.0      4
26.0      3
19.0      3
23.0      2
27.0      1
24.0      1
21.0      1
20.0      1
22.0      1
Name: count, dtype: int64

In [64]:
dataini['nbda'] = dataini['nbda'].fillna(0)

/tmp/ipykernel_5794/2940257307.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataini['nbda'] = dataini['nbda'].fillna(0)


In [65]:
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D,rea
0,8,9,0,01M37E,I671,2.0,76.0,4,1,0.0,l19,,0
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,,0


In [70]:
# on supprime les sujets décédés (modeSortie = 9)
# on enregistrer dans dataset à 'issue de la 1ere partie
dataset = dataini[dataini['modeSortie'] != 9].drop(['id', 'id_D'], axis = 1)

## 2. Feature engineering
### 2.1 Principes

In [71]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Normalizer

In [74]:
# on impute les valeursnulles par le mode
# ce n'est qu'une possibilité
SimpleImputer(strategy= 'most_frequent').fit_transform(dataset[['sexe']])

array([[2.0],
       [1.0],
       [1.0],
       ...,
       [1.0],
       [2.0],
       [1.0]], shape=(1322, 1), dtype=object)

In [78]:
# Encodage var quali
OneHotEncoder(drop= 'first').fit_transform(SimpleImputer(strategy= 'most_frequent').fit_transform(dataset[['sexe']])).toarray()

array([[1.],
       [0.],
       [0.],
       ...,
       [0.],
       [1.],
       [0.]], shape=(1322, 1))

### 2.2 Preprocessing pipeline

In [80]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [ ]:
features = dataset.drop(['rea'], axis = 1)      # X
label = dataset['rea']                          # Y

In [82]:
features.dtypes

modeEntree     object
modeSortie     object
duree           int32
ghm2           object
dp             object
sexe           object
age           float64
nbActe          int32
nbRum           int32
nbda          float64
dtype: object

In [ ]:
# def X quanti + récupération de l'index des var numériques
num_features = features.select_dtypes(['int32', 'float64']).columns
num_features

Index(['duree', 'age', 'nbActe', 'nbRum', 'nbda'], dtype='object')

In [84]:
# def X categorielles + récupération de l'index des var numériques
cat_features = features.select_dtypes(['object']).columns
cat_features

Index(['modeEntree', 'modeSortie', 'ghm2', 'dp', 'sexe'], dtype='object')

In [101]:
# Codage de transformers = enchainement de traitements et étapes
# transofrmation des var numériques
num_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer()),   # imputation de la moyenne
    ('scaler', StandardScaler())      # standardisation
])

num_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fea

In [124]:
# transofrmation des var numériques
cat_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer(strategy= 'most_frequent')),   # imputation du mode
    ('ohe', OneHotEncoder(handle_unknown= 'ignore'))         # dummy coding + on ignore les modalités non reconnues
]) 

cat_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('ohe', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If

In [125]:
# ici : possibilité d'ajouter des tranformations en termes carrés, cubiques, interctions, splines etc

In [126]:
preprocessor = ColumnTransformer(transformers= [                   # liste de transformers
    ('num', num_transformer, num_features),                        # nom de la liste de transformers
    ('cat', cat_transformer, cat_features)
])

In [127]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [128]:
preprocessor.fit_transform(features).toarray().shape

(1322, 112)

In [129]:
from sklearn.model_selection import train_test_split # fonction de découpage
X_train_val, X_test, y_train_val, y_test = train_test_split(features, label, 
                                                            test_size=0.1, 
                                                            random_state=42)    # 42 = réponse univerelle, le mythe du voyageur intergalactique :)
                                                        

## 3. Models
### 3.1 Random forest

In [130]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [157]:
# on instancialise
rf = RandomForestClassifier(random_state= 42)       # pour gérer le random, il vaut mieux fixer la graine

In [158]:
# on ajouter étape dans process
pip_rf = Pipeline(steps= [
    ('preproc', preprocessor),
    ('classifier', rf)
])

In [159]:
pip_rf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preproc', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [160]:
# pour récupérer les paramètres de la méthode
# pour RF : nb_estimators = nb d'arbres, max_depth = None signifie que l'arbre est maximal, max_feature : nb de var on regarde pour faire une branche
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [161]:
# dictionnaire avec les paramètres de RF
params_rf = {
    'classifier__n_estimators' : [100, 200],
    'classifier__max_depth' : [None, 10],
    'classifier__max_features' : ['sqrt', 'log2']
}

In [162]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [163]:
# préparation CV
cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 42)

In [164]:
metric_grid = ['accuracy', 'f1', 'roc_auc']

In [165]:
grid_rf = GridSearchCV(
    estimator = pip_rf,
    param_grid = params_rf,
    cv = cv,
    #scoring = 'accuracy',       # cherche le meilleur modèle selon le scoring de accuracy 
    scoring = metric_grid,        # cherche le meilleur modèle selon le scoring de accuracy, f1 ette roc_auc
    # refit= True                # et ensuite il réentraine sur tout le set de train+val ()=> pour après prédire sur l'éch test)
    refit = 'roc_auc'              # refit selon le meilleur modèle choisi selon roc_auc
)

In [166]:
grid_rf_fitted = grid_rf.fit(X_train_val, y_train_val)

In [167]:
# message d'erreur
# ValueError: Found unknown categories ['01K032', '01M121', '01M082'] in column 2 during transform
# des modalités dans l'échantillon test qui n'existent pas dans l'ech training => le modèle ne sait pas le prédire
# arrive très souvent => on l'ignore à l'étape d'entrainement du modèle
# on change dans cat_transformer : on ajoute l'option handle_unknown 

In [168]:
grid_rf_fitted.best_score_      # moyenne des 5 scores (accuracy ici) issus des 5 ech de validatio croisée

np.float64(0.876754599857615)

In [169]:
grid_rf_fitted.best_params_

{'classifier__max_depth': 10,
 'classifier__max_features': 'sqrt',
 'classifier__n_estimators': 100}

In [170]:
grid_rf_fitted.cv_results_.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_classifier__max_depth', 'param_classifier__max_features', 'param_classifier__n_estimators', 'params', 'split0_test_accuracy', 'split1_test_accuracy', 'split2_test_accuracy', 'split3_test_accuracy', 'split4_test_accuracy', 'mean_test_accuracy', 'std_test_accuracy', 'rank_test_accuracy', 'split0_test_f1', 'split1_test_f1', 'split2_test_f1', 'split3_test_f1', 'split4_test_f1', 'mean_test_f1', 'std_test_f1', 'rank_test_f1', 'split0_test_roc_auc', 'split1_test_roc_auc', 'split2_test_roc_auc', 'split3_test_roc_auc', 'split4_test_roc_auc', 'mean_test_roc_auc', 'std_test_roc_auc', 'rank_test_roc_auc'])

In [171]:
grid_rf_fitted.score(X_test, y_test)        # on applique le modèle sélectionné sur l'éch test

0.8857493857493858

In [172]:
# équivalent à 
accuracy_score(y_test, grid_rf_fitted.predict(X_test))

0.9548872180451128

### 3.4 Gradient boosting

In [194]:
from sklearn.ensemble import GradientBoostingClassifier

In [221]:
gb = GradientBoostingClassifier()
# n_estimators : ici c'est le nb d'itérations
# learning_rate : lambda
# n_iter_no_change : early stopping - attention ne pas mettre None - lié à tol : 0,001 (change minimal pour être considéé comme change)

In [222]:
gb.get_params()

{'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'log_loss',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': None,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [219]:
# on ajoute étape dans process
pip_gb = Pipeline(steps= [
    ('preproc', preprocessor),
    ('classifier', gb)
])

pip_gb

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preproc', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [211]:
params_gb = {
    'classifier__n_estimators' : [100, 200],
    'classifier__learning_rate' : [0.05, 0.1, 0.2],
    'classifier__n_iter_no_change' : [10]
}

In [214]:
grid_gb =  GridSearchCV(
    estimator = pip_gb,
    param_grid = params_gb,
    cv = cv,
    # scoring = 'accuracy',       # cherche le meilleur modèle selon le scoring de accuracy 
    scoring = metric_grid,        # cherche le meilleur modèle selon le scoring de accuracy, f1 ette roc_auc
    # refit= True                # et ensuite il réentraine sur tout le set de train+val ()=> pour après prédire sur l'éch test)
    refit = 'roc_auc',              # refit selon le meilleur modèle choisi selon roc_auc
    n_jobs= -1  # mobiliser tous les coeurs dispo
)

In [215]:
grid_gb_fitted = grid_gb.fit(X_train_val, y_train_val)

In [216]:
grid_gb_fitted.best_params_

{'classifier__learning_rate': 0.1,
 'classifier__n_estimators': 200,
 'classifier__n_iter_no_change': 10}

In [217]:
grid_gb_fitted.best_score_

np.float64(0.8557063630116394)

In [218]:
grid_gb_fitted.best_params_

{'classifier__learning_rate': 0.1,
 'classifier__n_estimators': 200,
 'classifier__n_iter_no_change': 10}

## 4. Model selection pipeline

In [223]:
models = [
    ('rf', rf, params_rf),
    ('gb', gb, params_gb)
]

In [224]:
preproc_list = [
    ('basic', preprocessor)
]

In [242]:
results = []

global_best_estimator = None        # on veut stocker le meilleur estimateur au fur et à mesure des boucles
global_best_score = -float('inf')   # on veut stocker le meilleur score au fur et à mesure des boucles

for preproc_id, preproc_object in preproc_list :
    for model_id, model_object, model_params in models :
        print(f'Model {model_id} with preprocessor {preproc_id}')
        
        pip = Pipeline(steps= [
            ('preproc', preproc_object),
            ('classifier' , model_object)
        ])

        grid = GridSearchCV(
            estimator= pip,
            cv = cv,
            param_grid= model_params,
            scoring= 'roc_auc',
            refit = True,            # par défaut
            n_jobs = -1
        )

        grid_fitted = grid.fit(X_train_val, y_train_val)

        cv_score = grid_fitted.best_score_

        if cv_score > global_best_score :                   # on met à jour le global_best_estimateur et le global_best_score si on trouve un meilleur estimateur
            global_best_score = cv_score
            global_best_estimator = grid_fitted.best_estimator_

        results.append(
            {
                'preprocessor' : preproc_id,
                'model' : model_id,
                'best_param' : grid_fitted.best_params_,
                'best_score' : grid_fitted.best_score_,
                'best_prediction' : grid_fitted.score(X_test, y_test)
            }
        )

Model rf with preprocessor basic
Model gb with preprocessor basic


In [243]:
results

[{'preprocessor': 'basic',
  'model': 'rf',
  'best_param': {'classifier__max_depth': 10,
   'classifier__max_features': 'sqrt',
   'classifier__n_estimators': 100},
  'best_score': np.float64(0.876754599857615),
  'best_prediction': 0.8857493857493858},
 {'preprocessor': 'basic',
  'model': 'gb',
  'best_param': {'classifier__learning_rate': 0.2,
   'classifier__n_estimators': 100,
   'classifier__n_iter_no_change': 10},
  'best_score': np.float64(0.8641900076008117),
  'best_prediction': 0.9113431613431614}]

In [244]:
pd.DataFrame(results)

,preprocessor,model,best_param,best_score,best_prediction
0,basic,rf,"{'classifier__max_depth': 10, 'classifier__max...",0.876755,0.885749
1,basic,gb,"{'classifier__learning_rate': 0.2, 'classifier...",0.864190,0.911343


In [245]:
# ré-entraîner sur l'ensemble des données
global_best_estimator.fit(features, label)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preproc', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 7. Model persistence

In [248]:
import joblib
joblib.dump(global_best_estimator, 'best_model_avc.pkl')

['best_model_avc.pkl']

In [249]:
model_loaded = joblib.load('best_model_avc.pkl')
model_loaded.predict

<bound method Pipeline.predict of Pipeline(steps=[('preproc',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['duree', 'age', 'nbActe', 'nbRum', 'nbda'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
      